In [2]:
from datasets import load_dataset

dataset = load_dataset('wikitext', name = 'wikitext-2-raw-v1', split='train')

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [13]:
def get_training_corpus():
    for i in range(0, len(dataset), 1000):
        yield dataset[i:i+1000]['text']

In [21]:
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers, processors, decoders
from tokenizers import Regex

tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))

tokenizer.normalizer = normalizers.Sequence([
        normalizers.Replace(Regex(r"[\p{Other}&[^\n\t\r]]"), ""), normalizers.Replace(Regex(r"[\s]"), " "),
        normalizers.Lowercase(),
        normalizers.NFD(), normalizers.StripAccents()
])

In [8]:
pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.WhitespaceSplit(),
    pre_tokenizers.Punctuation(),
])

In [11]:
special_tokens = ["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
trainer = trainers.WordPieceTrainer(
    vocab_size=52000,
    special_tokens=special_tokens,
)

In [14]:
tokenizer.train_from_iterator(get_training_corpus(), trainer = trainer)

In [15]:
cls_token_id = tokenizer.token_to_id("[CLS]")
sep_token_id = tokenizer.token_to_id("[SEP]")

In [19]:
tokenizer.post_processor = processors.TemplateProcessing(
    single=f"[CLS]:0 $A:0 [SEP]:0",
    pair=f"[CLS]:0 $A:0 [SEP]:0 $B:1 [SEP]:1",
    special_tokens=[("[CLS]", cls_token_id), ("[SEP]", sep_token_id)],
)

In [22]:
tokenizer.decoder = decoders.WordPiece(prefix = '##')